# Academic OSPO Readiness Survey — Analysis Notebook

This notebook produces the results in two parts, **walking the same questions in the same order as the spreadsheet tabs** (`analysis-helper.xlsx`): Respondents → Importance → Maturity by area → Level → Support needs by area → Service gaps → Experience & roles → Where support comes from → Open comments.

**Part A — Completed responses.** Uses only fully completed responses, so its numbers match the spreadsheet. Use it to reconcile the two tools.

**Part B — Progressive view.** Recalculates the *same* sections, in the *same* order, over everyone who reached each part — finished or not. A long survey loses people along the way, so "84 people told us why open source matters" is a stronger signal than "12 finished everything." Each Part B figure is labelled with its own group size, so its numbers are larger than Part A by design.

**Charts appear in both parts.** Every quantitative section saves an individual figure (prefixed `A_…` in Part A, `B_…` in Part B) so any chart can be dropped straight into a report.

**How to run:** set `INPUT_PATH` below and run all cells. Export from LimeSurvey as **All responses** (Question code and question text, Strip HTML on, Full answers).

In [ ]:
import os, datetime
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

INPUT_PATH = "your-survey-export.xlsx"   # <-- set this to your exported .xlsx file

plt.rcParams.update({"figure.dpi":110,"savefig.dpi":130,"axes.spines.top":False,"axes.spines.right":False,"font.size":10}); plt.rcParams["figure.max_open_warning"]=0
RUN = os.path.join("output", datetime.datetime.now().strftime("%Y-%m-%d_%H%M%S"))
os.makedirs(RUN, exist_ok=True)
def save(fig, name): fig.savefig(os.path.join(RUN, name), bbox_inches="tight")
print("Outputs will be written to:", RUN)

## Mappings
Each item is tagged with its OSMM level and its primary Academic OSPO service, matching `../instructions/40-interpret.md`.

In [ ]:
mat=[("Q300a","L2",1),("Q300b","L2",4),("Q300c","L2",6),("Q300d","L2",13),("Q310a","L2",4),("Q310b","L2",4),("Q310c","L2",5),("Q310d","L4",7),("Q310e","L2",4),("Q310f","L2",13),("Q320a","L2",5),("Q320b","L2",5),("Q320c","L2",5),("Q330a","L2",6),("Q330b","L2",8),("Q330c","L2",6),("Q330d","L2",6),("Q330e","L2",5),("Q330f","L2",4),("Q340a","L3",8),("Q340b","L3",8),("Q340c","L3",8),("Q340d","L3",8),("Q340e","L3",3),("Q350a","L3",8),("Q350b","L3",10),("Q350c","L4",10),("Q350d","L3",12),("Q350e","L3",12),("Q360a","L4",7),("Q360b","L4",3),("Q360c","L4",10),("Q360d","L4",10),("Q360e","L4",2),("Q360f","L4",10),("Q360g","L4",11),("Q360h","L4",11),("Q370a","L3",8),("Q370b","L3",8),("Q370c","L5",1),("Q370d","L5",1),("Q370e","L5",9),("Q370f","L4",9),("Q370g","L4",9)]
sup=[("Q400a",4),("Q400b",5),("Q400c",5),("Q400d",5),("Q400e",9),("Q400f",13),("Q410a",6),("Q410b",6),("Q410c",6),("Q420a",3),("Q420b",7),("Q420c",3),("Q420d",10),("Q430a",2),("Q430b",2),("Q430c",2),("Q430d",2),("Q440a",9),("Q440b",9),("Q440c",9),("Q440d",9),("Q450a",11),("Q450b",11),("Q450c",11),("Q450d",11),("Q450e",11),("Q450f",11),("Q450g",11),("Q450h",11),("Q460a",8),("Q460b",12),("Q460c",12),("Q460d",12),("Q460e",10)]
def addcodes(items):
    out=[];cnt={}
    for it in items:
        b=it[0][:4];cnt[b]=cnt.get(b,0)+1;out.append((it,f"{b}[SQ{cnt[b]:03d}]"))
    return out
MAT=[(c,lv,sv,ec) for (c,lv,sv),ec in addcodes(mat)]
SUP=[(c,sv,ec) for (c,sv),ec in addcodes(sup)]
SVC={1:"OSPO Strategy & Vision",2:"Community of Practice",3:"Project Lifecycle Support & Maintenance",4:"License & IPR Compliance",5:"Supply-Chain Security & SBOM",6:"Choosing, Using & Procuring Open Source",7:"Open-Sourcing Guidance",8:"Internal Advocacy & Buy-In",9:"Sustainability & Funding Support",10:"Impact Measurement",11:"Developer Tooling & Infrastructure",12:"Outreach & Events",13:"Open Source AI Policy"}
AREA=[("Q300","Policy & OSPO"),("Q310","Compliance and risk"),("Q320","Security & supply chain"),("Q330","Selection, use & procurement"),("Q340","Open source contribution"),("Q350","Community and outreach"),("Q360","Hosting and maintenance"),("Q370","Leadership and strategy")]
SUPAREA=[("Q400","Legal, compliance & risk"),("Q410","Selection, use & procurement"),("Q420","Project creation & maintenance"),("Q430","Community & mentoring"),("Q440","Funding & sustainability"),("Q450","Tools & compute resources"),("Q460","Advocacy & outreach")]
WRITEINS=[("Q10[comment]","Unit / department"),("Q301.","Where support lives — Policy & OSPO"),("Q311.","Where support lives — Compliance and risk"),("Q321.","Where support lives — Security & supply chain"),("Q331.","Where support lives — Selection, use & procurement"),("Q341.","Where support lives — Contribution"),("Q351.","Where support lives — Community & outreach"),("Q361.","Where support lives — Hosting & maintenance"),("Q371.","Where support lives — Leadership & strategy"),("Q401.","Other support — Legal, compliance & risk"),("Q411.","Other support — Selection, use & procurement"),("Q421.","Other support — Project creation & maintenance"),("Q431.","Other support — Community & mentoring"),("Q441.","Other support — Funding & sustainability"),("Q451.","Other support — Tools & compute"),("Q461.","Other support — Advocacy & outreach"),("Q70.","Final feedback"),("Q21[other]","Other reason it matters (Q21)"),("Q52[other]","Other ways you participate (Q52)"),("Q54[other]","Other contributor role (Q54)"),("Q55[other]","Other reason you participate (Q55)"),("Q56[other]","Other project category (Q56)"),("Q58.","Other challenges (Q58)")]

## Load the export and define the groups

In [ ]:
df=pd.read_excel(INPUT_PATH); df.columns=[str(c) for c in df.columns]; cols=list(df.columns)
def find(prefix): return next((c for c in cols if c.startswith(prefix)), None)
N=len(df)
subm=find("submitdate")
def reached(prefix):
    col=find(prefix)
    return pd.Series([False]*N, index=df.index) if col is None else (df[col].notna() & ~df[col].astype(str).str.strip().isin(["","N/A"]))

# Measure how far each person got by the PAGE they reached (lastpage) and whether
# they submitted (submitdate) -- never by whether answer cells are filled. The
# Section 3-5 questions are optional, so a skipped question is simply blank; counting
# by the page reached (not which boxes are ticked) keeps a deliberate skip counted as
# a real response rather than a missed section. Page map for this survey:
#   page 3 = confidence (end of Sections 1-2) | page 12 = last maturity question
#   (Section 3) | page 20 = last support question (Section 4) | page 25 = submitted.
lp=find("lastpage"); lpv=pd.to_numeric(df[lp],errors="coerce") if lp else pd.Series([np.nan]*N,index=df.index)
m_complete = (df[subm].notna() & (df[subm].astype(str).str.strip()!="") & (df[subm].astype(str)!="N/A")) if subm else pd.Series([True]*N, index=df.index)
m_p12 = lpv>=3      # completed Sections 1-2 (reached the confidence question)
m_p3  = lpv>=12     # completed Section 3 (reached the last maturity question)
m_p4  = lpv>=20     # completed Section 4 (reached the last support question)
m_p5  = lpv>=21     # reached Section 5 (saw the Q51 opt-in question)

dfc=df[m_complete]; d12=df[m_p12]; d3=df[m_p3]; d4=df[m_p4]; d5=df[m_p5]
Nc,N12,N3,N4,N5 = len(dfc),len(d12),len(d3),len(d4),len(d5)
print(f"Loaded {N} responses. Completed parts 1-2: {N12} | completed part 3: {N3} | completed part 4: {N4} | finished survey: {Nc}")

from decimal import Decimal, ROUND_HALF_UP
def rhalf(x, nd=0):
    # round half away from zero, matching the spreadsheet's ROUND() / "0" number format
    if x is None: return None
    if isinstance(x,float) and x!=x: return x  # NaN
    return float(Decimal(str(x)).quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP))
def pct(code, frame):
    col=find(code); n=len(frame)
    return None if (col is None or n==0) else int(rhalf((frame[col]=="Yes").sum()/n*100,0))
def maturity_df(frame):
    return pd.DataFrame([{"code":c,"level":lv,"svc":sv,"area":c[:4],"pct":pct(ec,frame)} for c,lv,sv,ec in MAT])
def by_level(mdf): return mdf.groupby("level")["pct"].mean().apply(lambda x: rhalf(x,0))
def by_area(mdf): return pd.Series({name:rhalf(mdf[mdf.area==code].pct.mean(),0) for code,name in AREA})
def support_area(frame): return pd.Series({name:rhalf(np.mean([pct(ec,frame) for c,sv,ec in SUP if c[:4]==code]),0) for code,name in SUPAREA})
def demand_svc(frame):
    s=pd.DataFrame([{"svc":sv,"pct":pct(ec,frame)} for c,sv,ec in SUP])
    return pd.Series({k:(s[s.svc==k].pct.mean() if (s.svc==k).any() else np.nan) for k in SVC})
def maturity_svc(mdf):
    return pd.Series({k:(mdf[mdf.svc==k].pct.mean() if (mdf.svc==k).any() else np.nan) for k in SVC})
def dist(prefix,labels,frame):
    col=find(prefix); vc=frame[col].value_counts() if col else {}
    return pd.Series({l:int(vc.get(l,0)) for l in labels})
# Roles (Q10) are read straight from the data: institutions are told to customise Q10 in setup,
# so we count whatever answer labels actually appear instead of matching a fixed list.
def role_counts(prefix, frame):
    col=find(prefix)
    if not col: return pd.Series(dtype=int)
    s=frame[col].dropna().astype(str).str.strip()
    s=s[~s.isin(["","N/A"])]
    vc=s.value_counts(); vc.index.name=None
    return vc
IMP=["Not important","Somewhat important","Important","Critical"]
CONF=["Not confident","Somewhat confident","Confident","Very confident"]

## Section renderers (shared by Part A and Part B)

In [ ]:
# ---- Shared section renderers: each prints a table and saves chart(s); used by BOTH parts ----
COL_ROLE="#4C78A8"; COL_IMP="#F58518"; COL_CONF="#72B7B2"; COL_MAT="#54A24B"; COL_DEM="#E45756"; COL_WHY="#9D755D"; COL_S5="#B279A2"
_q51=find("Q51.")
def _yes(code, frame):
    col=find(code); return 0 if col is None else int((frame[col].astype(str).str.strip()=="Yes").sum())

REASONS=[("Q21[SQ001]","Reduces vendor lock-in"),("Q21[SQ002]","Lowers costs"),("Q21[SQ003]","Enables collaboration"),("Q21[SQ004]","Transparency and trust"),("Q21[SQ005]","Accelerates innovation"),("Q21[SQ006]","Funder/regulatory requirements"),("Q21[SQ007]","Digital sovereignty / autonomy")]
SEC5=[
 ("Ways people participate (Q52)",[("Follow updates and discussions","Q52[SQ001]"),("Use open source applications","Q52[SQ002]"),("Use open source dependencies","Q52[SQ003]"),("Participate in development","Q52[SQ004]")]),
 ("Which activities you take part in (Q53)",[("Contribute code","Q53[SQ001]"),("Contribute documentation","Q53[SQ002]"),("Maintain a project","Q53[SQ003]"),("Report / document bugs","Q53[SQ004]"),("Offer ideas for features","Q53[SQ005]"),("Organisational / admin functions","Q53[SQ006]"),("Grow a project","Q53[SQ007]"),("Develop open source policy","Q53[SQ008]")]),
 ("Contributor roles ever held (Q54)",[("Maintainer","Q54[SQ001]"),("Contributor","Q54[SQ002]"),("Bug / issue reporter","Q54[SQ003]"),("Supervisor","Q54[SQ004]"),("IT / systems administrator","Q54[SQ005]"),("UI / UX designer","Q54[SQ006]"),("Community manager","Q54[SQ007]"),("Technical support","Q54[SQ008]"),("Educator","Q54[SQ009]")]),
 ("Why you participate (Q55)",[("Work duties require it","Q55[SQ001]"),("Helps other work duties","Q55[SQ002]"),("Improve tools in my field","Q55[SQ003]"),("Customise tools for my needs","Q55[SQ004]"),("Build a network of peers","Q55[SQ005]"),("Give back to the community","Q55[SQ006]"),("Improve my skills","Q55[SQ007]"),("Because it's fun","Q55[SQ008]")]),
 ("Project categories participated in (Q56)",[("Hardware","Q56[SQ001]"),("Applications","Q56[SQ002]"),("Plug-ins / extensions","Q56[SQ003]"),("Libraries / packages / frameworks","Q56[SQ004]"),("Automation scripts","Q56[SQ005]"),("Website code","Q56[SQ006]")]),
 ("Challenges encountered (Q57)",[("Limited time for new code","Q57[SQ001]"),("Limited time for documentation","Q57[SQ002]"),("Managing issues and PRs","Q57[SQ003]"),("Attracting users / contributors","Q57[SQ004]"),("Receiving recognition","Q57[SQ005]"),("Finding / hiring personnel","Q57[SQ006]"),("Managing security risks","Q57[SQ007]"),("Finding a community of peers","Q57[SQ008]"),("Finding mentors","Q57[SQ009]"),("Finding time to learn","Q57[SQ010]"),("Identifying learning resources","Q57[SQ011]"),("Navigating licensing / legal","Q57[SQ012]"),("Identifying funding sources","Q57[SQ013]"),("Securing funding","Q57[SQ014]")]),
]

# write-in key groups, matching the spreadsheet's distributed sheets
WI_UNIT=["Q10[comment]"]; WI_WHY=["Q21[other]"]
WI_SUPPORT=["Q401.","Q411.","Q421.","Q431.","Q441.","Q451.","Q461."]
WI_SEC5=["Q52[other]","Q54[other]","Q55[other]","Q56[other]","Q58."]
WI_SOURCE=["Q301.","Q311.","Q321.","Q331.","Q341.","Q351.","Q361.","Q371."]
WI_FINAL=["Q70."]; WI_LABEL=dict(WRITEINS)

def writeins(frame, keys, header):
    print(f"\n--- {header} (verbatim) ---"); found=False
    for k in keys:
        col=find(k)
        if not col: continue
        vals=[str(v).strip() for v in frame[col] if pd.notna(v) and str(v).strip() not in ("","N/A")]
        if vals:
            found=True; print(f"  {WI_LABEL.get(k,k)}  ({len(vals)})")
            for v in vals: print("    -", v)
    if not found: print("  (none in this group)")

def sec5_charts(frame, optin_n, suffix, prefix):
    for t,opts in SEC5:
        ser=pd.Series({l:(round(_yes(c,frame)/optin_n*100) if optin_n else 0) for l,c in opts}).sort_values()
        fig,ax=plt.subplots(figsize=(7,0.7+0.4*len(opts))); ser.plot.barh(ax=ax,color=COL_S5)
        ax.set_title(f"{t} — {suffix}",fontsize=11); ax.set_xlim(0,100); ax.set_xlabel("%")
        plt.tight_layout(); save(fig, f"{prefix}_{t.split('(')[-1].rstrip(') ')}.png"); plt.show()

def sec_respondents(frame, tag, prefix):
    role=role_counts("Q10.",frame); conf=dist("Q22.",CONF,frame)
    print(f"\n=== Respondents ({tag}) ===\nRole:\n",role.to_string(),"\n\nConfidence:\n",conf.to_string())
    fig,ax=plt.subplots(figsize=(6.4,3.2)); (role[role>0] if len(role) else role).plot.barh(ax=ax,color=COL_ROLE); ax.set_title(f"Role ({tag})"); plt.tight_layout(); save(fig,f"{prefix}_role.png"); plt.show()
    fig,ax=plt.subplots(figsize=(5.2,3.2)); conf.plot.bar(ax=ax,color=COL_CONF); ax.set_title(f"Confidence ({tag})"); plt.tight_layout(); save(fig,f"{prefix}_confidence.png"); plt.show()

def sec_importance(frame, tag, prefix):
    imp=dist("Q20.",IMP,frame); why=pd.Series({lab:pct(code,frame) for code,lab in REASONS})
    print(f"\n=== Importance ({tag}) ===\nImportance:\n",imp.to_string(),"\n\nWhy it matters (% of group):\n",why.to_string())
    fig,ax=plt.subplots(figsize=(5.2,3.2)); imp.plot.bar(ax=ax,color=COL_IMP); ax.set_title(f"Importance ({tag})"); plt.tight_layout(); save(fig,f"{prefix}_importance.png"); plt.show()
    fig,ax=plt.subplots(figsize=(7,3.2)); why.sort_values().plot.barh(ax=ax,color=COL_WHY); ax.set_title(f"Why it matters (%, {tag})"); ax.set_xlim(0,100); plt.tight_layout(); save(fig,f"{prefix}_why.png"); plt.show()
    return imp

def sec_maturity_area(mdf, tag, prefix):
    ba=by_area(mdf); print(f"\n=== Maturity by area ({tag}) ===\n",ba.to_string())
    fig,ax=plt.subplots(figsize=(7,3.6)); ba.sort_values().plot.barh(ax=ax,color=COL_MAT); ax.set_title(f"Maturity by area ({tag})"); ax.set_xlim(0,100); plt.tight_layout(); save(fig,f"{prefix}_maturity_area.png"); plt.show()
    return ba

def sec_level(mdf, tag, prefix):
    bl=by_level(mdf); print(f"\n=== Maturity by OSMM level ({tag}) ===\n",bl.to_string())
    fig,ax=plt.subplots(figsize=(6.2,3.4)); bl.plot.bar(ax=ax,color=COL_MAT); ax.set_title(f"Maturity by level ({tag})"); ax.set_ylim(0,100); ax.set_ylabel("%"); plt.tight_layout(); save(fig,f"{prefix}_maturity_level.png"); plt.show()
    return bl

def sec_support_area(frame, tag, prefix):
    sa=support_area(frame); print(f"\n=== Support needs by area ({tag}) ===\n",sa.to_string())
    fig,ax=plt.subplots(figsize=(7,3.4)); sa.sort_values().plot.barh(ax=ax,color=COL_DEM); ax.set_title(f"Support needs by area ({tag})"); ax.set_xlim(0,100); plt.tight_layout(); save(fig,f"{prefix}_support_area.png"); plt.show()
    return sa

def sec_service_gaps(mat_mdf, dem_frame, tag_m, tag_d, prefix):
    dem=demand_svc(dem_frame); mat=maturity_svc(mat_mdf)   # raw means
    gap_raw=dem-mat                                          # gap from raw, as the sheet does
    svc=pd.DataFrame({f"maturity ({tag_m})":mat.apply(lambda x: rhalf(x,0)),f"demand ({tag_d})":dem.apply(lambda x: rhalf(x,0))})
    svc.index=[SVC[i] for i in svc.index]; svc["gap"]=[rhalf(g,0) for g in gap_raw]
    print(f"\n=== Service gaps — demand vs maturity ===\n",svc.sort_values("gap",ascending=False).to_string())
    d=svc.dropna(subset=[svc.columns[1]]).sort_values(svc.columns[1])
    fig,ax=plt.subplots(figsize=(7,4)); ax.barh(d.index,d.iloc[:,1],color=COL_DEM); ax.set_title(f"Service demand ({tag_d})"); ax.set_xlim(0,100); plt.tight_layout(); save(fig,f"{prefix}_demand.png"); plt.show()
    q=svc.dropna(subset=[svc.columns[0],svc.columns[1]]).rename(columns={svc.columns[0]:"maturity",svc.columns[1]:"demand"})
    if len(q):
        fig,ax=plt.subplots(figsize=(10,7)); ax.scatter(q["maturity"],q["demand"],s=80,color=COL_ROLE,zorder=3)
        ax.axvline(q["maturity"].median(),color="#bbb",ls="--"); ax.axhline(q["demand"].median(),color="#bbb",ls="--"); ax.margins(0.18)
        ax.set_xlabel("Maturity %"); ax.set_ylabel("Demand %"); ax.set_title(f"Priority quadrant ({tag_d}) — upper-left = act first")
        labels=[ax.text(r["maturity"],r["demand"],nm,fontsize=8) for nm,r in q.iterrows()]
        try:
            from adjustText import adjust_text
            adjust_text(labels,ax=ax,expand=(1.5,2.0),force_text=(0.6,1.0),arrowprops=dict(arrowstyle="-",color="#888",lw=0.6))
        except Exception: print("adjustText not installed; labels may overlap. pip install adjustText")
        save(fig,f"{prefix}_quadrant.png"); plt.show()
    return svc

def sec_experience(frame, tag, prefix):
    optin = frame[frame[_q51].astype(str).str.strip()=="Yes"] if _q51 else frame.iloc[0:0]
    nopt=len(optin); ng=len(frame)
    print(f"\n=== Community experience & roles — Section 5 ({tag}) ===")
    if _q51:
        y=int((frame[_q51].astype(str).str.strip()=="Yes").sum()); no=int((frame[_q51].astype(str).str.strip()=="No").sum())
        print(f"Willing (Q51): Yes {y} ({round(y/ng*100) if ng else 0}%), No {no}")
    print(f"{nopt} opted in; percentages below are of those {nopt}.\n")
    for t,opts in SEC5:
        print(f"## {t}")
        for lab,code in opts:
            c=_yes(code,frame); p=round(c/nopt*100) if nopt else 0
            print(f"  {lab}: {c}  ({p}%)")
        print()
    sec5_charts(frame, nopt, f"{tag}, opted-in n={nopt}", f"{prefix}_s5")
    return nopt

# Part A — Completed responses (matches the spreadsheet)

Everything below uses only the **fully completed** responses and is laid out in the **spreadsheet's tab order**, so the numbers reconcile with `analysis-helper.xlsx`.

In [ ]:
print(f"Part A — {Nc} fully completed responses (of {N} in the file).")
mat_cols=[find(ec) for _,_,_,ec in MAT if find(ec)]; sup_cols=[find(ec) for _,_,ec in SUP if find(ec)]
my=(dfc[mat_cols]=="Yes").sum(axis=1); sy=(dfc[sup_cols]=="Yes").sum(axis=1)
ticks_all=int(((my==len(mat_cols))|(sy==len(sup_cols))).sum()); ticks_none=int(((my==0)&(sy==0)).sum())
print(f"Data-quality flags: ticked everything {ticks_all}, ticked nothing throughout {ticks_none}, of {Nc} completed")
TA=f"completed, n={Nc}"; mA=maturity_df(dfc)

In [ ]:
# Tabs 1-2: Respondents, Importance
sec_respondents(dfc, TA, "A"); writeins(dfc, WI_UNIT, "Respondents — unit / department")
impA=sec_importance(dfc, TA, "A"); writeins(dfc, WI_WHY, "Importance — other reasons")

In [ ]:
# Tabs 3-4: Maturity by area, Level summary
baA=sec_maturity_area(mA, TA, "A"); blA=sec_level(mA, TA, "A")

In [ ]:
# Tab 5: Support needs by area
saA=sec_support_area(dfc, TA, "A"); writeins(dfc, WI_SUPPORT, "Support needs — other support wanted")

In [ ]:
# Tab 6: Service gaps (demand vs maturity)
svcA=sec_service_gaps(mA, dfc, TA, TA, "A")

In [ ]:
# Tab 9: Experience & roles (Section 5)
sec_experience(dfc, TA, "A"); writeins(dfc, WI_SEC5, "Experience & roles — other (Section 5)")

In [ ]:
# Tab 10: Where support comes from (Section 3 free-text)
writeins(dfc, WI_SOURCE, "Where support comes from")

In [ ]:
# Tab 11: Open comments (end of survey)
writeins(dfc, WI_FINAL, "Open comments — final feedback")

# Part B — Progressive view (everyone who reached each part)

The **same sections in the same order**, recalculated over everyone who reached the relevant part — finished or not. Numbers are larger than Part A on purpose, and each figure is labelled with the group it is based on. Group per section: Respondents & Importance use everyone who completed parts 1–2; Maturity uses the part‑3 group; Support needs & Service gaps use the part‑4 group; Experience & roles uses everyone who reached Section 5. (Final feedback is only saved on submit, so Open comments stays the finishers.)

In [ ]:
funnel=[("Completed parts 1-2",N12),("Completed part 3 (maturity)",N3),("Completed part 4 (support)",N4),("Reached Section 5",N5),("Finished the survey",Nc)]
for lab,v in funnel: print(f"  {lab:30s} {v}")
fig,ax=plt.subplots(figsize=(6.5,2.6)); ax.barh([f[0] for f in funnel][::-1],[f[1] for f in funnel][::-1],color="#4C78A8")
ax.set_xlabel("respondents"); ax.set_title("How far respondents got"); plt.tight_layout(); save(fig,"B_funnel.png"); plt.show()

In [ ]:
# Tabs 1-2: Respondents, Importance  (parts 1-2 group)
TB12=f"parts 1-2, n={N12}"
sec_respondents(d12, TB12, "B"); writeins(d12, WI_UNIT, "Respondents — unit / department (parts 1-2)")
impB=sec_importance(d12, TB12, "B"); writeins(d12, WI_WHY, "Importance — other reasons (parts 1-2)")

In [ ]:
# Tabs 3-4: Maturity by area, Level summary  (part-3 group)
TB3=f"part-3 group, n={N3}"; mB=maturity_df(d3)
baB=sec_maturity_area(mB, TB3, "B"); blB=sec_level(mB, TB3, "B")

In [ ]:
# Tab 5: Support needs by area  (part-4 group)
TB4=f"part-4 group, n={N4}"
saB=sec_support_area(d4, TB4, "B"); writeins(d4, WI_SUPPORT, "Support needs — other support wanted (part-4)")

In [ ]:
# Tab 6: Service gaps  (maturity from part-3, demand from part-4)
svcB=sec_service_gaps(mB, d4, f"n={N3}", f"n={N4}", "B")

In [ ]:
# Tab 9: Experience & roles (Section 5)  (everyone who reached Section 5)
TB5=f"reached S5, n={N5}"
sec_experience(d5, TB5, "B"); writeins(d5, WI_SEC5, "Experience & roles — other (reached S5)")

In [ ]:
# Tab 10: Where support comes from  (part-3 group)
writeins(d3, WI_SOURCE, "Where support comes from (part-3 group)")

In [ ]:
# Tab 11: Open comments  (finishers — only saved on submit)
writeins(dfc, WI_FINAL, "Open comments — final feedback (finishers)")

## Save tables and findings

In [ ]:
with pd.ExcelWriter(os.path.join(RUN,"summary_tables.xlsx")) as xw:
    blA.rename("maturity %").to_frame().to_excel(xw,sheet_name="A_by_level")
    baA.rename("maturity %").to_frame().to_excel(xw,sheet_name="A_by_area")
    svcA.to_excel(xw,sheet_name="A_services")
    blB.rename("maturity %").to_frame().to_excel(xw,sheet_name="B_by_level")
    svcB.to_excel(xw,sheet_name="B_services")
topA=svcA.dropna(subset=["gap"]).sort_values("gap",ascending=False).head(3)
lines=[f"# Findings — {os.path.basename(INPUT_PATH)}","",
 "## Part A — completed responses only (matches the spreadsheet)",
 f"- Respondents: {Nc} completed (of {N} in the file)",
 f"- Data-quality flags: ticks-everything {ticks_all}, ticks-nothing {ticks_none}",
 f"- Maturity by level: "+", ".join(f"{k} {v:.0f}%" for k,v in blA.items()),
 f"- Most mature area: {baA.idxmax()} ({baA.max():.0f}%); least: {baA.idxmin()} ({baA.min():.0f}%)",
 "- Priority services (largest gap): "+", ".join(f"{i} (gap {r.gap:.0f})" for i,r in topA.iterrows()),
 "",
 "## Part B — progressive view (more of your data)",
 f"- Completed parts 1-2: {N12} | completed part 3: {N3} | completed part 4: {N4} | finished: {Nc}",
 f"- Importance among the {N12} who did parts 1-2: "+", ".join(f"{k} {v}" for k,v in impB.items()),
 f"- Maturity by level among the {N3} who did part 3: "+", ".join(f"{k} {v:.0f}%" for k,v in blB.items()),
 f"- Top demand among the {N4} who did part 4: "+", ".join(f"{i} {r.iloc[1]:.0f}%" for i,r in svcB.dropna(subset=[svcB.columns[1]]).sort_values(svcB.columns[1],ascending=False).head(3).iterrows())]
open(os.path.join(RUN,"findings.md"),"w").write("\n".join(lines))
print("\n".join(lines)); print("\nSaved to",RUN,"->",sorted(os.listdir(RUN)))